# SAM 2 - Advertising Board Segmentation

This notebook evaluates SAM 2 for advertising-board segmentation in football images.

Two experiments are considered:

1. Ground-truth bounding boxes → SAM 2 → predicted masks
2. RF-DETR predicted bounding boxes → SAM 2 → predicted masks

The first experiment isolates the segmentation capability of SAM 2 when an accurate spatial prompt is available.

The second experiment evaluates the complete RF-DETR → SAM 2 pipeline.

The manually segmented masks are used as pixel-level ground truth. Because the segmentation dataset may contain incomplete annotations, both quantitative and qualitative analyses are considered.

PARTE 1 — ANALISI DATASET

In [1]:
from pathlib import Path
import sys
import re
import json
import random
import time

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DIR = Path(
        "/content/drive/MyDrive/football-adboard-segmentation"
    )

    # Cerchiamo il dataset in entrambe le posizioni possibili
    candidates = [
        PROJECT_DIR / "datasets" / "segmentation",
        PROJECT_DIR / "data" / "raw" / "segmentation",
    ]

    SEGMENTATION_DIR = next(
        (p for p in candidates if p.exists()),
        candidates[0]
    )

else:
    PROJECT_DIR = Path(
        "/Users/marcodalbis/Desktop/Deep Learning/"
        "football-adboard-segmentation"
    )

    SEGMENTATION_DIR = (
        PROJECT_DIR / "data" / "raw" / "segmentation"
    )

IMAGE_DIR = (
    SEGMENTATION_DIR
    / "Tagged_Images"
    / "Tagged Images"
)

MASK_DIR = (
    SEGMENTATION_DIR
    / "Masks"
    / "Masks"
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "sam2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("PROJECT_DIR:", PROJECT_DIR)
print("SEGMENTATION_DIR:", SEGMENTATION_DIR)
print("Images:", IMAGE_DIR.exists())
print("Masks :", MASK_DIR.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB: True
PROJECT_DIR: /content/drive/MyDrive/football-adboard-segmentation
SEGMENTATION_DIR: /content/drive/MyDrive/football-adboard-segmentation/datasets/segmentation
Images: True
Masks : True


In [2]:
image_files = sorted(IMAGE_DIR.glob("*.jpg"))
mask_files = sorted(MASK_DIR.glob("*.png"))

print("Numero immagini:", len(image_files))
print("Numero maschere :", len(mask_files))

Numero immagini: 1620
Numero maschere : 1620


In [3]:
def extract_id(path):
    match = re.search(r"(\d+)", path.stem)
    return int(match.group(1)) if match else None


images_by_id = {
    extract_id(path): path
    for path in image_files
}

masks_by_id = {
    extract_id(path): path
    for path in mask_files
}

image_ids = set(images_by_id)
mask_ids = set(masks_by_id)

common_ids = image_ids & mask_ids
images_without_mask = image_ids - mask_ids
masks_without_image = mask_ids - image_ids

print("Immagini totali         :", len(image_ids))
print("Maschere totali          :", len(mask_ids))
print("Coppie immagine-maschera :", len(common_ids))
print("Immagini senza maschera  :", len(images_without_mask))
print("Maschere senza immagine  :", len(masks_without_image))

Immagini totali         : 1620
Maschere totali          : 1620
Coppie immagine-maschera : 1620
Immagini senza maschera  : 0
Maschere senza immagine  : 0


PARTE 2 — CONTROLLO DELLE MASCHERE

In [4]:
def load_binary_mask(path, threshold=127):
    mask = np.array(Image.open(path))

    if mask.ndim == 3:
        binary = np.any(mask > threshold, axis=2)
    else:
        binary = mask > threshold

    return binary.astype(bool)

In [ ]:
shape_mismatches = []
empty_masks = []
mask_value_set = set()
foreground_percentages = []

for idx in sorted(common_ids):

    image = np.array(
        Image.open(images_by_id[idx]).convert("RGB")
    )

    raw_mask = np.array(
        Image.open(masks_by_id[idx])
    )

    binary_mask = load_binary_mask(masks_by_id[idx])

    if image.shape[:2] != binary_mask.shape[:2]:
        shape_mismatches.append(idx)

    if not binary_mask.any():
        empty_masks.append(idx)

    mask_value_set.update(
        np.unique(raw_mask).tolist()
    )

    foreground_percentages.append(
        100 * binary_mask.mean()
    )

print("Coppie controllate:", len(common_ids))
print("Shape mismatch:", len(shape_mismatches))
print("Maschere vuote:", len(empty_masks))

print("\nValori presenti nelle mask:")
print(sorted(mask_value_set))

print("\nForeground:")
print(f"min    : {np.min(foreground_percentages):.3f}%")
print(f"media  : {np.mean(foreground_percentages):.3f}%")
print(f"mediana: {np.median(foreground_percentages):.3f}%")
print(f"max    : {np.max(foreground_percentages):.3f}%")

In [ ]:
import cv2
from collections import Counter

component_counts = []
component_areas = []

for idx in sorted(common_ids):

    mask = load_binary_mask(
        masks_by_id[idx]
    ).astype(np.uint8)

    num_labels, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(
            mask,
            connectivity=8
        )
    )

    # Il label 0 è lo sfondo
    n_components = num_labels - 1

    component_counts.append(n_components)

    if n_components > 0:
        component_areas.extend(
            stats[1:, cv2.CC_STAT_AREA].tolist()
        )

print("Distribuzione numero componenti:")
print(Counter(component_counts))

print("\nNumero medio componenti per frame:",
      np.mean(component_counts))

print("Area componenti:")
print("min     :", np.min(component_areas))
print("mediana :", np.median(component_areas))
print("media   :", np.mean(component_areas))
print("max     :", np.max(component_areas))

In [ ]:
sample_ids = [0, 1, 2, 100, 500, 1000]

for idx in sample_ids:

    image = np.array(
        Image.open(images_by_id[idx]).convert("RGB")
    )

    mask = load_binary_mask(
        masks_by_id[idx]
    )

    overlay = image.copy()

    overlay[mask] = (
        0.5 * overlay[mask]
        + 0.5 * np.array([255, 0, 0])
    ).astype(np.uint8)

    fig, axes = plt.subplots(
        1, 3,
        figsize=(18, 6)
    )

    axes[0].imshow(image)
    axes[0].set_title(f"Frame {idx}")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Ground-truth mask")
    axes[1].axis("off")

    axes[2].imshow(overlay)
    axes[2].set_title("GT overlay")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

PARTE 3 — RICAVARE LE GT BOX DALLE MASCHERE

In [ ]:
def get_component_boxes(binary_mask, min_area=1):

    mask_uint8 = binary_mask.astype(np.uint8)

    num_labels, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(
            mask_uint8,
            connectivity=8
        )
    )

    boxes = []

    for label_id in range(1, num_labels):

        x = stats[label_id, cv2.CC_STAT_LEFT]
        y = stats[label_id, cv2.CC_STAT_TOP]
        w = stats[label_id, cv2.CC_STAT_WIDTH]
        h = stats[label_id, cv2.CC_STAT_HEIGHT]
        area = stats[label_id, cv2.CC_STAT_AREA]

        if area < min_area:
            continue

        x2 = x + w - 1
        y2 = y + h - 1

        boxes.append(
            np.array(
                [x, y, x2, y2],
                dtype=np.float32
            )
        )

    return boxes

In [ ]:
MIN_COMPONENT_AREA = 1

In [ ]:
from matplotlib.patches import Rectangle

for idx in [1, 100, 500]:

    image = np.array(
        Image.open(images_by_id[idx]).convert("RGB")
    )

    gt_mask = load_binary_mask(
        masks_by_id[idx]
    )

    boxes = get_component_boxes(
        gt_mask,
        min_area=MIN_COMPONENT_AREA
    )

    fig, ax = plt.subplots(
        figsize=(14, 7)
    )

    ax.imshow(image)

    for box in boxes:

        x1, y1, x2, y2 = box

        rect = Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            linewidth=2
        )

        ax.add_patch(rect)

    ax.set_title(
        f"Frame {idx} - "
        f"{len(boxes)} GT box"
    )

    ax.axis("off")

    plt.show()

PARTE 4 — INSTALLAZIONE SAM 2

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponibile:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip install -q \
    "git+https://github.com/facebookresearch/sam2.git" \
    huggingface_hub \
    "rfdetr==1.9.1" \
    supervision

In [ ]:
import torch
from sam2.sam2_image_predictor import SAM2ImagePredictor

DEVICE = "cuda"

SAM2_MODEL_ID = "facebook/sam2.1-hiera-small"

sam_predictor = (
    SAM2ImagePredictor.from_pretrained(
        SAM2_MODEL_ID,
        device=DEVICE
    )
)

print("SAM 2 caricato")
print("Model:", SAM2_MODEL_ID)
print("Device:", sam_predictor.device)

PARTE 5 — FUNZIONI SAM 2

In [ ]:
def compute_segmentation_metrics(
    predicted_mask,
    ground_truth_mask
):

    pred = predicted_mask.astype(bool)
    gt = ground_truth_mask.astype(bool)

    intersection = np.logical_and(
        pred, gt
    ).sum()

    union = np.logical_or(
        pred, gt
    ).sum()

    pred_area = pred.sum()
    gt_area = gt.sum()

    iou = (
        intersection / union
        if union > 0
        else 1.0
    )

    dice = (
        2 * intersection
        / (pred_area + gt_area)
        if (pred_area + gt_area) > 0
        else 1.0
    )

    return {
        "iou": float(iou),
        "dice": float(dice),
        "intersection": int(intersection),
        "union": int(union),
        "pred_area": int(pred_area),
        "gt_area": int(gt_area),
    }

In [ ]:
from contextlib import nullcontext


def get_autocast_context():

    if DEVICE != "cuda":
        return nullcontext()

    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    return torch.autocast(
        device_type="cuda",
        dtype=dtype
    )


def sam_masks_from_boxes(
    image,
    boxes
):

    if len(boxes) == 0:
        return []

    predicted_masks = []

    with torch.inference_mode(), get_autocast_context():

        sam_predictor.set_image(image)

        for box in boxes:

            masks, scores, logits = (
                sam_predictor.predict(
                    box=np.asarray(
                        box,
                        dtype=np.float32
                    ),
                    multimask_output=False
                )
            )

            predicted_masks.append(
                masks[0].astype(bool)
            )

    return predicted_masks


def union_masks(
    masks,
    image_shape
):

    result = np.zeros(
        image_shape[:2],
        dtype=bool
    )

    for mask in masks:
        result |= mask.astype(bool)

    return result

ESPERIMENTO A — GT BOX → SAM 2

Test 1 sola immagine


In [ ]:
TEST_ID = 100

image = np.array(
    Image.open(
        images_by_id[TEST_ID]
    ).convert("RGB")
)

gt_mask = load_binary_mask(
    masks_by_id[TEST_ID]
)

gt_boxes = get_component_boxes(
    gt_mask,
    min_area=MIN_COMPONENT_AREA
)

sam_masks = sam_masks_from_boxes(
    image,
    gt_boxes
)

predicted_mask = union_masks(
    sam_masks,
    image.shape
)

metrics = compute_segmentation_metrics(
    predicted_mask,
    gt_mask
)

print("Frame:", TEST_ID)
print("GT boxes:", len(gt_boxes))
print("IoU :", metrics["iou"])
print("Dice:", metrics["dice"])

In [ ]:
fig, axes = plt.subplots(
    1, 4,
    figsize=(22, 6)
)

axes[0].imshow(image)
axes[0].set_title("Original image")

axes[1].imshow(gt_mask, cmap="gray")
axes[1].set_title("Ground truth")

axes[2].imshow(
    predicted_mask,
    cmap="gray"
)
axes[2].set_title("SAM 2 prediction")

overlay = image.copy()

overlay[predicted_mask] = (
    0.5 * overlay[predicted_mask]
    + 0.5 * np.array([255, 0, 0])
).astype(np.uint8)

axes[3].imshow(overlay)

axes[3].set_title(
    f"SAM 2 overlay\n"
    f"IoU={metrics['iou']:.3f} "
    f"Dice={metrics['dice']:.3f}"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
random.seed(42)

SMOKE_IDS = random.sample(
    sorted(common_ids),
    20
)

smoke_results = []

for idx in SMOKE_IDS:

    image = np.array(
        Image.open(
            images_by_id[idx]
        ).convert("RGB")
    )

    gt_mask = load_binary_mask(
        masks_by_id[idx]
    )

    gt_boxes = get_component_boxes(
        gt_mask,
        min_area=MIN_COMPONENT_AREA
    )

    sam_masks = sam_masks_from_boxes(
        image,
        gt_boxes
    )

    predicted_mask = union_masks(
        sam_masks,
        image.shape
    )

    metrics = compute_segmentation_metrics(
        predicted_mask,
        gt_mask
    )

    smoke_results.append({
        "frame_id": idx,
        "num_gt_boxes": len(gt_boxes),
        "iou": metrics["iou"],
        "dice": metrics["dice"],
    })


smoke_df = pd.DataFrame(
    smoke_results
)

display(smoke_df)

print("\nMean IoU :",
      smoke_df["iou"].mean())

print("Mean Dice:",
      smoke_df["dice"].mean())

Completato esperimento A

In [ ]:
from tqdm.auto import tqdm

gt_box_results = []

total_intersection = 0
total_union = 0
total_pred_area = 0
total_gt_area = 0

start_time = time.time()

for idx in tqdm(
    sorted(common_ids),
    desc="GT box -> SAM 2"
):

    image = np.array(
        Image.open(
            images_by_id[idx]
        ).convert("RGB")
    )

    gt_mask = load_binary_mask(
        masks_by_id[idx]
    )

    gt_boxes = get_component_boxes(
        gt_mask,
        min_area=MIN_COMPONENT_AREA
    )

    sam_masks = sam_masks_from_boxes(
        image,
        gt_boxes
    )

    predicted_mask = union_masks(
        sam_masks,
        image.shape
    )

    metrics = compute_segmentation_metrics(
        predicted_mask,
        gt_mask
    )

    gt_box_results.append({
        "frame_id": idx,
        "num_gt_boxes": len(gt_boxes),
        "iou": metrics["iou"],
        "dice": metrics["dice"],
    })

    total_intersection += metrics[
        "intersection"
    ]

    total_union += metrics["union"]

    total_pred_area += metrics[
        "pred_area"
    ]

    total_gt_area += metrics[
        "gt_area"
    ]


gt_box_df = pd.DataFrame(
    gt_box_results
)

elapsed = time.time() - start_time

print(
    f"Tempo totale: "
    f"{elapsed / 60:.1f} minuti"
)

risultati esperimento A

In [ ]:
gt_mean_iou = gt_box_df["iou"].mean()
gt_median_iou = gt_box_df["iou"].median()

gt_mean_dice = gt_box_df["dice"].mean()
gt_median_dice = gt_box_df["dice"].median()

gt_micro_iou = (
    total_intersection / total_union
)

gt_micro_dice = (
    2 * total_intersection
    / (total_pred_area + total_gt_area)
)

print("===== GT BOX -> SAM 2 =====")

print(
    f"Mean IoU    : {gt_mean_iou:.4f}"
)

print(
    f"Median IoU  : {gt_median_iou:.4f}"
)

print(
    f"Micro IoU   : {gt_micro_iou:.4f}"
)

print(
    f"Mean Dice   : {gt_mean_dice:.4f}"
)

print(
    f"Median Dice : {gt_median_dice:.4f}"
)

print(
    f"Micro Dice  : {gt_micro_dice:.4f}"
)

In [ ]:
GT_RESULTS_DIR = (
    OUTPUT_DIR / "gt_box_sam2"
)

GT_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

gt_box_df.to_csv(
    GT_RESULTS_DIR
    / "per_image_metrics.csv",
    index=False
)

gt_summary = {
    "model": SAM2_MODEL_ID,
    "num_images": len(gt_box_df),
    "mean_iou": float(gt_mean_iou),
    "median_iou": float(gt_median_iou),
    "micro_iou": float(gt_micro_iou),
    "mean_dice": float(gt_mean_dice),
    "median_dice": float(gt_median_dice),
    "micro_dice": float(gt_micro_dice),
}

with (
    GT_RESULTS_DIR
    / "summary.json"
).open("w") as f:

    json.dump(
        gt_summary,
        f,
        indent=4
    )

print("Risultati salvati in:")
print(GT_RESULTS_DIR)

ESPERIMENTO B — RF-DETR → SAM 2

Carico RF DETR

In [ ]:
from rfdetr import RFDETRNano

CHECKPOINT_PATH = (
    PROJECT_DIR
    / "outputs"
    / "rfdetr_nano_smoke_test_local"
    / "checkpoint_best_total.pth"
)

print("Checkpoint:")
print(CHECKPOINT_PATH)
print("Esiste:", CHECKPOINT_PATH.exists())

assert CHECKPOINT_PATH.exists(), (
    "Checkpoint RF-DETR non trovato"
)

detector = RFDETRNano.from_checkpoint(
    str(CHECKPOINT_PATH)
)

print("RF-DETR caricato")

In [ ]:
RF_THRESHOLD = 0.50

TEST_ID = 100

image = np.array(
    Image.open(
        images_by_id[TEST_ID]
    ).convert("RGB")
)

detections = detector.predict(
    image,
    threshold=RF_THRESHOLD
)

print(
    "Detection trovate:",
    len(detections)
)

for box, conf in zip(
    detections.xyxy,
    detections.confidence
):
    print(
        "bbox:",
        np.round(box, 1),
        "confidence:",
        round(float(conf), 3)
    )

In [ ]:
fig, ax = plt.subplots(
    figsize=(14, 7)
)

ax.imshow(image)

for box, conf in zip(
    detections.xyxy,
    detections.confidence
):

    x1, y1, x2, y2 = box

    rect = Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=2
    )

    ax.add_patch(rect)

    ax.text(
        x1,
        y1,
        f"{conf:.2f}",
        fontsize=10
    )

ax.set_title(
    "RF-DETR predictions "
    "on segmentation dataset"
)

ax.axis("off")

plt.show()

Gestione del problema delle GT incomplete

Qui facciamo due valutazioni.

blind: utilizziamo tutte le detection RF-DETR. È la pipeline realmente automatica.

GT-aligned: per la valutazione quantitativa consideriamo solo le detection spazialmente associate alle regioni annotate nella mask. Questo evita di penalizzare RF-DETR/SAM 2 quando segmentano un pannello reale che però è assente dalla GT.

In [ ]:
def intersection_over_pred_box(
    pred_box,
    gt_box
):

    px1, py1, px2, py2 = pred_box
    gx1, gy1, gx2, gy2 = gt_box

    ix1 = max(px1, gx1)
    iy1 = max(py1, gy1)

    ix2 = min(px2, gx2)
    iy2 = min(py2, gy2)

    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)

    intersection = iw * ih

    pred_area = max(
        (px2 - px1) * (py2 - py1),
        1e-9
    )

    return intersection / pred_area


def get_gt_aligned_indices(
    pred_boxes,
    gt_boxes,
    min_overlap=0.25
):

    selected = []

    for i, pred_box in enumerate(pred_boxes):

        px1, py1, px2, py2 = pred_box

        cx = (px1 + px2) / 2
        cy = (py1 + py2) / 2

        keep = False

        for gt_box in gt_boxes:

            gx1, gy1, gx2, gy2 = gt_box

            center_inside = (
                gx1 <= cx <= gx2
                and
                gy1 <= cy <= gy2
            )

            overlap = (
                intersection_over_pred_box(
                    pred_box,
                    gt_box
                )
            )

            if (
                center_inside
                or overlap >= min_overlap
            ):
                keep = True
                break

        if keep:
            selected.append(i)

    return selected

In [ ]:
TEST_ID = 100

image = np.array(
    Image.open(
        images_by_id[TEST_ID]
    ).convert("RGB")
)

gt_mask = load_binary_mask(
    masks_by_id[TEST_ID]
)

gt_boxes = get_component_boxes(
    gt_mask,
    min_area=MIN_COMPONENT_AREA
)

detections = detector.predict(
    image,
    threshold=RF_THRESHOLD
)

rf_boxes = [
    np.asarray(box, dtype=np.float32)
    for box in detections.xyxy
]

rf_sam_masks = sam_masks_from_boxes(
    image,
    rf_boxes
)

# Pipeline completamente automatica
blind_mask = union_masks(
    rf_sam_masks,
    image.shape
)

# Detection corrispondenti alla GT disponibile
aligned_indices = (
    get_gt_aligned_indices(
        rf_boxes,
        gt_boxes
    )
)

aligned_sam_masks = [
    rf_sam_masks[i]
    for i in aligned_indices
]

aligned_mask = union_masks(
    aligned_sam_masks,
    image.shape
)

blind_metrics = (
    compute_segmentation_metrics(
        blind_mask,
        gt_mask
    )
)

aligned_metrics = (
    compute_segmentation_metrics(
        aligned_mask,
        gt_mask
    )
)

print("RF-DETR boxes:", len(rf_boxes))

print(
    "GT-aligned boxes:",
    len(aligned_indices)
)

print("\nBLIND:")
print(blind_metrics)

print("\nGT-ALIGNED:")
print(aligned_metrics)

In [ ]:
fig, axes = plt.subplots(
    1, 4,
    figsize=(22, 6)
)

axes[0].imshow(image)
axes[0].set_title("Original")

axes[1].imshow(
    gt_mask,
    cmap="gray"
)
axes[1].set_title("Ground truth")

axes[2].imshow(
    blind_mask,
    cmap="gray"
)

axes[2].set_title(
    "RF-DETR → SAM 2\n"
    f"IoU={blind_metrics['iou']:.3f}"
)

axes[3].imshow(
    aligned_mask,
    cmap="gray"
)

axes[3].set_title(
    "GT-aligned RF-DETR → SAM 2\n"
    f"IoU={aligned_metrics['iou']:.3f}"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

ESPERIMENTO B COMPLETO
Cella 28 — Tutte le 1.620 immagini

In [ ]:
pipeline_results = []

blind_intersection = 0
blind_union = 0
blind_pred_area = 0
blind_gt_area = 0

aligned_intersection = 0
aligned_union = 0
aligned_pred_area = 0
aligned_gt_area = 0

start_time = time.time()

for idx in tqdm(
    sorted(common_ids),
    desc="RF-DETR -> SAM 2"
):

    image = np.array(
        Image.open(
            images_by_id[idx]
        ).convert("RGB")
    )

    gt_mask = load_binary_mask(
        masks_by_id[idx]
    )

    gt_boxes = get_component_boxes(
        gt_mask,
        min_area=MIN_COMPONENT_AREA
    )

    detections = detector.predict(
        image,
        threshold=RF_THRESHOLD
    )

    rf_boxes = [
        np.asarray(
            box,
            dtype=np.float32
        )
        for box in detections.xyxy
    ]

    rf_sam_masks = (
        sam_masks_from_boxes(
            image,
            rf_boxes
        )
    )

    # -----------------------
    # BLIND / FULL PIPELINE
    # -----------------------

    blind_mask = union_masks(
        rf_sam_masks,
        image.shape
    )

    blind_metrics = (
        compute_segmentation_metrics(
            blind_mask,
            gt_mask
        )
    )

    # -----------------------
    # GT-ALIGNED DIAGNOSTIC
    # -----------------------

    aligned_indices = (
        get_gt_aligned_indices(
            rf_boxes,
            gt_boxes
        )
    )

    aligned_masks = [
        rf_sam_masks[i]
        for i in aligned_indices
    ]

    aligned_mask = union_masks(
        aligned_masks,
        image.shape
    )

    aligned_metrics = (
        compute_segmentation_metrics(
            aligned_mask,
            gt_mask
        )
    )

    pipeline_results.append({
        "frame_id": idx,
        "num_gt_boxes": len(gt_boxes),
        "num_rf_boxes": len(rf_boxes),
        "num_aligned_boxes": len(
            aligned_indices
        ),
        "blind_iou":
            blind_metrics["iou"],
        "blind_dice":
            blind_metrics["dice"],
        "aligned_iou":
            aligned_metrics["iou"],
        "aligned_dice":
            aligned_metrics["dice"],
    })

    blind_intersection += (
        blind_metrics["intersection"]
    )

    blind_union += (
        blind_metrics["union"]
    )

    blind_pred_area += (
        blind_metrics["pred_area"]
    )

    blind_gt_area += (
        blind_metrics["gt_area"]
    )

    aligned_intersection += (
        aligned_metrics["intersection"]
    )

    aligned_union += (
        aligned_metrics["union"]
    )

    aligned_pred_area += (
        aligned_metrics["pred_area"]
    )

    aligned_gt_area += (
        aligned_metrics["gt_area"]
    )


pipeline_df = pd.DataFrame(
    pipeline_results
)

elapsed = time.time() - start_time

print(
    f"Tempo totale: "
    f"{elapsed / 60:.1f} minuti"
)

In [ ]:
blind_mean_iou = (
    pipeline_df["blind_iou"].mean()
)

blind_mean_dice = (
    pipeline_df["blind_dice"].mean()
)

blind_micro_iou = (
    blind_intersection
    / blind_union
)

blind_micro_dice = (
    2 * blind_intersection
    /
    (
        blind_pred_area
        + blind_gt_area
    )
)


aligned_mean_iou = (
    pipeline_df["aligned_iou"].mean()
)

aligned_mean_dice = (
    pipeline_df["aligned_dice"].mean()
)

aligned_micro_iou = (
    aligned_intersection
    / aligned_union
)

aligned_micro_dice = (
    2 * aligned_intersection
    /
    (
        aligned_pred_area
        + aligned_gt_area
    )
)


print("===== RF-DETR -> SAM 2 =====")

print("\nFULL / BLIND PIPELINE")
print(
    f"Mean IoU   : {blind_mean_iou:.4f}"
)
print(
    f"Micro IoU  : {blind_micro_iou:.4f}"
)
print(
    f"Mean Dice  : {blind_mean_dice:.4f}"
)
print(
    f"Micro Dice : {blind_micro_dice:.4f}"
)

print("\nGT-ALIGNED DIAGNOSTIC")
print(
    f"Mean IoU   : {aligned_mean_iou:.4f}"
)
print(
    f"Micro IoU  : {aligned_micro_iou:.4f}"
)
print(
    f"Mean Dice  : {aligned_mean_dice:.4f}"
)
print(
    f"Micro Dice : {aligned_micro_dice:.4f}"
)

PARTE 7 — CONFRONTO FINALE

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Experiment":
            "GT bbox → SAM 2",
        "Mean IoU":
            gt_mean_iou,
        "Micro IoU":
            gt_micro_iou,
        "Mean Dice":
            gt_mean_dice,
        "Micro Dice":
            gt_micro_dice,
    },
    {
        "Experiment":
            "RF-DETR → SAM 2 (blind)",
        "Mean IoU":
            blind_mean_iou,
        "Micro IoU":
            blind_micro_iou,
        "Mean Dice":
            blind_mean_dice,
        "Micro Dice":
            blind_micro_dice,
    },
    {
        "Experiment":
            "RF-DETR → SAM 2 (GT-aligned)",
        "Mean IoU":
            aligned_mean_iou,
        "Micro IoU":
            aligned_micro_iou,
        "Mean Dice":
            aligned_mean_dice,
        "Micro Dice":
            aligned_micro_dice,
    }
])

display(
    comparison_df.round(4)
)

In [ ]:
PIPELINE_RESULTS_DIR = (
    OUTPUT_DIR
    / "rfdetr_sam2"
)

PIPELINE_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

pipeline_df.to_csv(
    PIPELINE_RESULTS_DIR
    / "per_image_metrics.csv",
    index=False
)

comparison_df.to_csv(
    OUTPUT_DIR
    / "sam2_experiment_comparison.csv",
    index=False
)

final_summary = {
    "sam2_model":
        SAM2_MODEL_ID,

    "rf_detr_threshold":
        RF_THRESHOLD,

    "num_images":
        len(common_ids),

    "gt_bbox_sam2": {
        "mean_iou":
            float(gt_mean_iou),
        "micro_iou":
            float(gt_micro_iou),
        "mean_dice":
            float(gt_mean_dice),
        "micro_dice":
            float(gt_micro_dice),
    },

    "rfdetr_sam2_blind": {
        "mean_iou":
            float(blind_mean_iou),
        "micro_iou":
            float(blind_micro_iou),
        "mean_dice":
            float(blind_mean_dice),
        "micro_dice":
            float(blind_micro_dice),
    },

    "rfdetr_sam2_gt_aligned": {
        "mean_iou":
            float(aligned_mean_iou),
        "micro_iou":
            float(aligned_micro_iou),
        "mean_dice":
            float(aligned_mean_dice),
        "micro_dice":
            float(aligned_micro_dice),
    },
}

with (
    OUTPUT_DIR
    / "sam2_final_summary.json"
).open("w") as f:

    json.dump(
        final_summary,
        f,
        indent=4
    )

print("Risultati finali salvati in:")
print(OUTPUT_DIR)